#  ReAct Agents 

In [ ]:
# ============================================================
# 1. IMPORTS + OPENAI API KEY
# ============================================================

from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

# Load variables from .env
load_dotenv()

# Check whether OPENAI_API_KEY exists
if os.environ.get("OPENAI_API_KEY"):
    print("Bro API KEY Variable exists")
else:
    raise ValueError("OPENAI_API_KEY not found")


from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    AIMessage
)

from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import (
    StrOutputParser,
    PydanticOutputParser
)


# Create OpenAI LLM
llm_openai = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0
)


# ============================================================
# 2. TOOLS
# ============================================================

# ------------------------------------------------------------
# TOOL 1: DuckDuckGo Search Tool
# ------------------------------------------------------------

from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun(
    description="This is a tool to search the web for news"
)


# ------------------------------------------------------------
# TOOL 2: Wikipedia Search Tool
# ------------------------------------------------------------

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wikipedia_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(),
    description="This is a tool to search Wikipedia"
)


# ------------------------------------------------------------
# TOOL 3: Custom Enterprise Tool
# ------------------------------------------------------------

from langchain.tools import tool


@tool
def enterprise_tool(query: str) -> str:
    """
    This is a tool to send emails to employees
    """

    return "Email Sent"


# ============================================================
# 3. CREATE TOOLKIT
# ============================================================

ToolKit = [
    search_tool,
    wikipedia_tool,
    enterprise_tool
]

print(ToolKit)


# ============================================================
# 4. CREATE REACT AGENT
# ============================================================

from langchain.agents import create_agent

model = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0.1,
    max_tokens=1000,
    timeout=30
)

# Create agent using the model and all available tools
agent = create_agent(
    model,
    tools=ToolKit
)

print(agent)


# ============================================================
# 5. INVOKE REACT AGENT USING STREAMING
# ============================================================

example_query = "Give me the latest news about the stock market"

# Run the agent
events = agent.stream(
    {
        "messages": [
            ("user", example_query)
        ]
    },
    stream_mode="values",
)


# Print every event generated by the agent
for event in events:
    event["messages"][-1].pretty_print()


# ============================================================
# 6. WITHOUT BINDING TOOLS
# ============================================================

response = llm_openai.invoke(
    "What's the latest news about the stock market?"
)

print(response)


# ============================================================
# 7. MANUALLY BIND TOOLS TO THE LLM
# ============================================================

llm_binded = llm_openai.bind_tools(ToolKit)


# Now the LLM knows about the available tools
response = llm_binded.invoke(
    "What's the latest news about the stock market?"
)

print(response)